# STR 量稳换手率因子构造（预热集：2004–2005；训练集：2006–2017）

本 Notebook 只负责因子生产：

1. 从 `Data/stocks_daily_04-25.parquet` 读取 2004 年起的换手率、成交量和总市值；
2. 使用 2004–2005 年数据为 20 日滚动窗口预热；
3. 计算每只股票过去 20 个连续市场交易日换手率的样本标准差；
4. 只在 2006-01-04 至 2017-12-29 的正式训练区间做 MAD 去极值、对数总市值中性化和 Z-score；
5. 保存标准因子 Parquet，交给 `Back_test/通用因子回测模板.ipynb`。

> 原始方法按月末计算 STR；这里输出每个有效交易日的同定义因子，调仓频率由回测框架控制。预热数据只参与历史滚动，不进入训练因子文件，也不参与训练期横截面处理。全流程仅使用信号日及以前的信息。


## 1. 导入依赖与参数

基准参数为过去 20 个连续市场交易日，`MIN_PERIODS` 固定为 20。方法定义已完整写入 Notebook，运行不依赖外部研报文件。环境变量仅用于快速冒烟测试，不改变正式默认值。


In [ ]:
from pathlib import Path
from IPython.display import display
import os

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.parquet as pq

pd.set_option('display.max_columns', 40)
pd.set_option('display.float_format', lambda x: f'{x:.6f}')

PROJECT_DIR = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'Back_test').exists() and (p / 'Factor').exists()), Path.cwd())
DATA_PATH = PROJECT_DIR / 'Data' / 'stocks_daily_04-25.parquet'
NOTEBOOK_DIR = PROJECT_DIR / 'Factor'

#------------------------修改区
# 2004–2005 只用于滚动窗口预热；正式因子严格输出 2006–2017。
WARMUP_START_DATE = os.getenv(
    'TURNOVER_STABILITY_WARMUP_START_DATE',
    '2017-11-01'
)
START_DATE = os.getenv(
    'TURNOVER_STABILITY_START_DATE',
    '2018-01-02'
)
END_DATE = os.getenv(
    'TURNOVER_STABILITY_END_DATE',
    '2021-12-31'
)
OUTPUT_DIR = Path(os.getenv(
    'TURNOVER_STABILITY_OUTPUT_DIR',
    str(NOTEBOOK_DIR / 'turnover_stability_output_20180102_20211231')
))
#-------------------------

ROLLING_WINDOW = 20
MIN_PERIODS = ROLLING_WINDOW
MAD_SCALE = 3.0
MIN_NEUTRALIZE_STOCKS = 100
EXCLUDE_ST = True

WARMUP_START_TS = pd.Timestamp(WARMUP_START_DATE)
START_TS = pd.Timestamp(START_DATE)
END_TS = pd.Timestamp(END_DATE)
WARMUP_START_INT = int(WARMUP_START_TS.strftime('%Y%m%d'))
START_INT = int(START_TS.strftime('%Y%m%d'))
END_INT = int(END_TS.strftime('%Y%m%d'))
DATE_TAG = f'{START_TS:%Y%m%d}_{END_TS:%Y%m%d}'

assert WARMUP_START_TS < START_TS <= END_TS
assert DATA_PATH.exists(), f'找不到行情文件：{DATA_PATH}'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('行情文件：', DATA_PATH)
print('预热区间：', WARMUP_START_DATE, '至', (START_TS - pd.Timedelta(days=1)).date())
print('训练区间：', START_DATE, '至', END_DATE)
print('输出目录：', OUTPUT_DIR)
print('pandas / pyarrow：', pd.__version__, '/', pa.__version__)


## 2. 分批读取 Parquet

只读取因子计算所需字段，并在 Arrow 层过滤日期。读取范围从预热期起点开始，但不会载入训练期结束后的验证集和测试集数据。


In [ ]:
RAW_COLUMNS = [
    '股票代码', '股票名称', '交易日',
    '成交量（手）', '换手率（%）', '总市值（万元）'
]


def load_market_window(path, start_int, end_int, batch_size=300_000):
    parquet_file = pq.ParquetFile(path)
    selected = []
    scanned = 0

    for batch in parquet_file.iter_batches(
        batch_size=batch_size,
        columns=RAW_COLUMNS,
        use_threads=True,
    ):
        table = pa.Table.from_batches([batch])
        date_column = table['交易日']
        mask = pc.and_(
            pc.greater_equal(date_column, pa.scalar(start_int)),
            pc.less_equal(date_column, pa.scalar(end_int)),
        )
        filtered = table.filter(mask)
        if filtered.num_rows:
            selected.append(filtered)
        scanned += table.num_rows

    assert selected, '预热及训练区间内没有读取到数据'
    result = pa.concat_tables(selected).to_pandas()
    print(f'扫描 {scanned:,} 行，预热及训练区间保留 {len(result):,} 行')
    return result


market = load_market_window(DATA_PATH, WARMUP_START_INT, END_INT)
market = market.rename(columns={
    '股票代码': 'code',
    '股票名称': 'name',
    '交易日': 'date_int',
    '成交量（手）': 'volume',
    '换手率（%）': 'turnover_rate',
    '总市值（万元）': 'total_market_cap',
})
market['date'] = pd.to_datetime(market['date_int'], format='%Y%m%d')
market = market.drop(columns='date_int')

# 市场交易日序号用于检查每只股票的 20 日窗口是否真正连续。
market_dates = pd.Index(sorted(market['date'].dropna().unique()))
date_to_index = pd.Series(np.arange(len(market_dates)), index=market_dates)
market['date_index'] = market['date'].map(date_to_index)

# 去掉上市前等整行空记录；停牌或缺数窗口会在后面的严格检查中失效。
market = market.dropna(
    subset=['volume', 'turnover_rate', 'total_market_cap'], how='all'
)
market = market.sort_values(['code', 'date'], kind='stable').reset_index(drop=True)

assert market['date'].min() >= WARMUP_START_TS
assert market['date'].max() <= END_TS
assert (market['date'] < START_TS).any(), '没有读到正式训练期之前的预热数据'
assert not market.duplicated(['date', 'code']).any(), '存在重复的 股票-交易日 记录'

print('实际读取日期：', market['date'].min().date(), '至', market['date'].max().date())
print('预热行情行数：', f"{(market['date'] < START_TS).sum():,}")
print('训练行情行数：', f"{market['date'].between(START_TS, END_TS).sum():,}")
print('市场交易日：', f'{len(market_dates):,}')
print('股票数量：', f"{market['code'].nunique():,}")
display(market.head())


## 3. 数据质量与可投资样本

换手率必须是非负有限数、成交量必须大于 0、总市值必须大于 0。停牌和关键字段缺失的记录不进入滚动窗口；若 `EXCLUDE_ST=True`，名称中含 ST 的股票不进入当日横截面。质量检查覆盖预热期和训练期。


In [ ]:
for column in ['volume', 'turnover_rate', 'total_market_cap']:
    market[column] = pd.to_numeric(market[column], errors='coerce')

valid_turnover = (
    market['code'].notna()
    & np.isfinite(market['turnover_rate'])
    & (market['turnover_rate'] >= 0)
    & np.isfinite(market['volume'])
    & (market['volume'] > 0)
)
valid_market_cap = (
    np.isfinite(market['total_market_cap'])
    & (market['total_market_cap'] > 0)
)
market.loc[~valid_turnover, 'turnover_rate'] = np.nan
market['log_market_cap'] = np.log(
    market['total_market_cap'].where(valid_market_cap)
)

if EXCLUDE_ST:
    is_st = market['name'].fillna('').str.upper().str.contains('ST', regex=False)
else:
    is_st = pd.Series(False, index=market.index)

quality_summary = pd.Series({
    '记录行数': len(market),
    '有效换手率行数': int(valid_turnover.sum()),
    '换手率或成交量异常行数': int((~valid_turnover).sum()),
    '有效总市值行数': int(valid_market_cap.sum()),
    'ST记录数': int(is_st.sum()),
    '重复股票日期数': int(market.duplicated(['date', 'code']).sum()),
})
display(quality_summary.to_frame('数量'))

## 4. 计算 20 日原始 STR

定义：

```text
STR_RAW(i,t) = Std(换手率(i,t-19:t), ddof=1)
```

要求 20 个换手率全部有效，而且首尾市场交易日序号正好相差 19，避免停牌或缺失交易日时把更早的记录误塞进窗口。2004–2005 年行情先参与滚动预热，计算完成后才截取 2006–2017 年正式训练区间。


In [ ]:
grouped_by_code = market.groupby('code', sort=False)
market['rolling_valid_obs'] = grouped_by_code['turnover_rate'].transform(
    lambda values: values.rolling(ROLLING_WINDOW, min_periods=1).count()
)
market['turnover_stability_raw'] = grouped_by_code['turnover_rate'].transform(
    lambda values: values.rolling(
        ROLLING_WINDOW, min_periods=MIN_PERIODS
    ).std(ddof=1)
)

window_start_index = grouped_by_code['date_index'].shift(ROLLING_WINDOW - 1)
is_consecutive_window = (
    market['date_index'] - window_start_index
).eq(ROLLING_WINDOW - 1)
market.loc[~is_consecutive_window, 'turnover_stability_raw'] = np.nan

# ST 只在横截面处理前排除；不会修改历史换手率。
market.loc[is_st, 'turnover_stability_raw'] = np.nan

# 验证正式训练首日：具备连续完整窗口且可投资的记录必须已有因子值。
target_mask = market['date'].between(START_TS, END_TS)
assert target_mask.any(), '正式训练区间内没有行情数据'
first_training_date = market.loc[target_mask, 'date'].min()
first_day_mask = market['date'].eq(first_training_date)
first_day_eligible = (
    first_day_mask
    & ~is_st
    & market['rolling_valid_obs'].ge(ROLLING_WINDOW)
    & is_consecutive_window
)
assert market.loc[first_day_eligible, 'turnover_stability_raw'].notna().all(), (
    '训练首日仍存在由滚动计算造成的意外缺失'
)

warmup_diagnostics = pd.Series({
    '正式训练首日': first_training_date,
    '首日行情记录数': int(first_day_mask.sum()),
    '首日连续完整20期且可投资记录数': int(first_day_eligible.sum()),
    '首日有效原始因子数': int(
        market.loc[first_day_mask, 'turnover_stability_raw'].notna().sum()
    ),
    '首日历史不足或窗口不连续数': int(
        (first_day_mask & (
            market['rolling_valid_obs'].lt(ROLLING_WINDOW)
            | ~is_consecutive_window
        )).sum()
    ),
})
display(warmup_diagnostics.to_frame('值'))

# 从这里开始只保留正式训练期；预热记录不参与横截面处理和最终输出。
market = market.loc[target_mask].copy().reset_index(drop=True)
assert market['date'].between(START_TS, END_TS).all()

display(
    market.loc[
        market['turnover_stability_raw'].notna(),
        ['date', 'code', 'turnover_rate', 'rolling_valid_obs',
         'turnover_stability_raw']
    ].head()
)


## 5. 每日 MAD 去极值、市值中性化与标准化

只在正式训练区间逐日执行：

1. 中位数 ± `MAD_SCALE × MAD` 去极值；
2. 回归 `STR_winsor = 截距 + beta × log(总市值) + 残差`；
3. 对残差做 Z-score，得到 `turnover_stability_factor`。

方法要求市值中性化。当前行情表没有完整 Barra 风格和行业字段，因此这里只做基准定义所需的 Size 中性化，不额外做“纯净 STR”的十风格加行业回归。


In [ ]:
date_group = market.groupby('date', sort=False)['turnover_stability_raw']
daily_median = date_group.transform('median')
absolute_deviation = (market['turnover_stability_raw'] - daily_median).abs()
daily_mad = absolute_deviation.groupby(
    market['date'], sort=False
).transform('median')

lower_limit = daily_median - MAD_SCALE * daily_mad
upper_limit = daily_median + MAD_SCALE * daily_mad
clipped = market['turnover_stability_raw'].clip(
    lower=lower_limit, upper=upper_limit
)
market['turnover_stability_winsor'] = clipped.where(
    daily_mad > 0, market['turnover_stability_raw']
)

eligible = (
    market['turnover_stability_winsor'].notna()
    & market['log_market_cap'].notna()
)
x = market['log_market_cap'].where(eligible)
y = market['turnover_stability_winsor'].where(eligible)
daily_key = market['date']
daily_n = eligible.groupby(daily_key, sort=False).transform('sum')
x_mean = x.groupby(daily_key, sort=False).transform('mean')
y_mean = y.groupby(daily_key, sort=False).transform('mean')
dx = x - x_mean
dy = y - y_mean
covariance_numerator = (dx * dy).groupby(
    daily_key, sort=False
).transform('sum')
x_variance_numerator = dx.pow(2).groupby(
    daily_key, sort=False
).transform('sum')
valid_regression = (
    (daily_n >= MIN_NEUTRALIZE_STOCKS)
    & (x_variance_numerator > 0)
)
size_beta = (covariance_numerator / x_variance_numerator).where(
    valid_regression
)
market['turnover_stability_neutral'] = (
    dy - size_beta * dx
).where(valid_regression)

neutral_group = market.groupby(
    'date', sort=False
)['turnover_stability_neutral']
neutral_mean = neutral_group.transform('mean')
neutral_std = neutral_group.transform('std')
market['turnover_stability_factor'] = (
    (market['turnover_stability_neutral'] - neutral_mean)
    / neutral_std.where(neutral_std > 0)
)

# 中性化诊断：残差与 log(总市值) 的相关系数应接近 0。
y_sum_of_squares = dy.pow(2).groupby(
    daily_key, sort=False
).transform('sum')
raw_size_correlation = covariance_numerator / np.sqrt(
    x_variance_numerator * y_sum_of_squares
)
residual = market['turnover_stability_neutral']
residual_centered = residual - residual.groupby(
    daily_key, sort=False
).transform('mean')
residual_covariance = (dx * residual_centered).groupby(
    daily_key, sort=False
).transform('sum')
residual_sum_of_squares = residual_centered.pow(2).groupby(
    daily_key, sort=False
).transform('sum')
neutralized_size_correlation = residual_covariance / np.sqrt(
    x_variance_numerator * residual_sum_of_squares
)

market['_size_beta'] = size_beta
market['_raw_size_corr'] = raw_size_correlation
market['_neutralized_size_corr'] = neutralized_size_correlation
factor_coverage = market.groupby('date').agg(
    market_rows=('code', 'size'),
    valid_turnover=('turnover_rate', 'count'),
    raw_factor=('turnover_stability_raw', 'count'),
    neutralized_factor=('turnover_stability_neutral', 'count'),
    valid_factor=('turnover_stability_factor', 'count'),
    size_beta=('_size_beta', 'first'),
    raw_size_correlation=('_raw_size_corr', 'first'),
    neutralized_size_correlation=('_neutralized_size_corr', 'first'),
)
factor_coverage['coverage'] = (
    factor_coverage['valid_factor'] / factor_coverage['market_rows']
)

valid_neutral_corr = factor_coverage[
    'neutralized_size_correlation'
].dropna()
assert not valid_neutral_corr.empty, '没有可用的市值中性化结果'
assert valid_neutral_corr.abs().max() < 1e-7, '市值中性化检查失败'
display(factor_coverage.describe().T)

## 6. 保存标准因子文件

这是因子生产与通用回测的分界。输出仅包含正式训练期，保留原始因子、去极值因子、中性化残差和最终标准化因子；`asof_date` 表示每条记录只使用截至信号日的信息。


In [ ]:
factor_columns = [
    'date', 'code', 'name', 'turnover_rate',
    'turnover_stability_raw', 'turnover_stability_winsor',
    'turnover_stability_neutral', 'turnover_stability_factor'
]
factor_panel = market.loc[
    market['turnover_stability_factor'].notna(), factor_columns
].copy()
factor_panel['asof_date'] = factor_panel['date']

assert factor_panel['date'].between(START_TS, END_TS).all()
assert factor_panel['date'].min() == market['date'].min()
assert not factor_panel.duplicated(['date', 'code']).any()
assert np.isfinite(factor_panel['turnover_stability_factor']).all()

factor_path = OUTPUT_DIR / f'turnover_stability_factor_{DATE_TAG}.parquet'
factor_panel.to_parquet(factor_path, index=False, compression='zstd')
factor_coverage.to_csv(OUTPUT_DIR / 'factor_coverage.csv', index_label='date')

factor_summary = pd.Series({
    '因子记录数': len(factor_panel),
    '股票数量': factor_panel['code'].nunique(),
    '有效因子首日': factor_panel['date'].min(),
    '有效因子末日': factor_panel['date'].max(),
    '中性化后最大绝对市值相关': valid_neutral_corr.abs().max(),
    '输出大小MB': factor_path.stat().st_size / 1024 / 1024,
})
display(factor_summary.to_frame('值'))
print('因子文件：', factor_path)
print('覆盖率与中性化诊断：', OUTPUT_DIR / 'factor_coverage.csv')
print('下一步：运行', PROJECT_DIR / 'Back_test' / '通用因子回测模板.ipynb')


## 7. 接入通用回测

在 `Back_test/通用因子回测模板.ipynb` 中设置：

```python
FACTOR_NAME = 'turnover_stability'
FACTOR_COLUMN = 'turnover_stability_factor'
FACTOR_DIRECTION = 'negative'
FACTOR_PREPROCESSED = True
FACTOR_SOURCE = (
    LEARNING_DIR / 'Factor' / 'turnover_stability_output_20060104_20171229'
    / 'turnover_stability_factor_20060104_20171229.parquet'
)
```

最终列已经完成 MAD、Size 中性化和 Z-score，因此回测中的 `NEUTRALIZE` 保持 `False`，避免重复中性化。原始方法采用月末调仓；若要严格对照，还需在回测层使用月末信号，而不是把 `REBALANCE_STEP=20` 简单等同于月末。
